# Distilling the Knowledge in a Neural Network

For tasks like speech and object recognition, training must extract structure from very large, highly redundant datasets,
but it does not need to operate in real time; deployment to a large number of users has much more stringent requirements on latency and computational resources.
We can therefore train a very cumbersome model, such as an ensemble of models or a single very large model trained with a strong regularizer,
and then use a different kind of training, which we call **distillation**, to transfer the knowledge from the cumbersome model to a small model that is more suitable for deployment.

This notebook follows Hinton, Vinyals, and Dean (2015). It has eight sections:

| Section | Content |
|---|---|
| 1 | The data |
| 2 | Softmax with a temperature: a worked example |
| 3 | The distillation objective and its gradient: a worked example, checked with automatic differentiation |
| 4 | Training the cumbersome model |
| 5 | Distillation versus training on hard targets |
| 6 | A much smaller net: intermediate temperatures work best |
| 7 | Omitting the digit 3 from the transfer set |
| 8 | Summary |

**How to run**: execute the cells from top to bottom. A GPU is faster ("Runtime → Change runtime type → T4 GPU");
on a CPU the whole notebook takes a few minutes.

## 1. The data

We use a subset of MNIST: 10,000 training cases (1000 of each digit) and the full test set of 10,000 cases.
Each case is a 28×28 grayscale image with pixel values from 0 to 255. The file is `kd_mnist.npz` (about 3.3 MB).

The data can be obtained in two ways, and the next cell checks which one applies:

1. **Drag the file into Colab**: put `kd_mnist.npz` in the "Files" panel on the left, that is, in the default folder `/content`.
2. **Read it from the GitHub repository of this module**: if the file is not in the panel, it is downloaded automatically.

In [ ]:
import os
import urllib.request

import numpy as np

np.set_printoptions(suppress=True)  # print decimals without scientific notation, as on the slides

DATA = 'kd_mnist.npz'
URL = ('https://raw.githubusercontent.com/pychang-ai/115-1_DLMA/main/'
       'topics/knowledge-distillation/kd_mnist.npz')

if os.path.exists(DATA):
    print('Using the uploaded', DATA)
else:
    print(DATA, 'not found; downloading it from the GitHub repository')
    urllib.request.urlretrieve(URL, DATA)

d = np.load(DATA)
x_train = d['x_train'].reshape(-1, 784).astype('float32') / 255.0
y_train = d['y_train'].astype('int32')
x_test = d['x_test'].reshape(-1, 784).astype('float32') / 255.0
y_test = d['y_test'].astype('int32')
print('training cases', x_train.shape, 'test cases', x_test.shape)
print('training cases per digit', np.bincount(y_train))
print('test cases per digit    ', np.bincount(y_test))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for k, ax in enumerate(axes):
    i = np.where(y_train == k)[0][0]
    ax.imshow(x_train[i].reshape(28, 28), cmap='gray_r')
    ax.set_title(str(k))
    ax.axis('off')
plt.show()

## 2. Softmax with a temperature

Neural networks typically produce class probabilities by using a "softmax" output layer that converts the logit, $z_i$,
computed for each class into a probability, $q_i$, by comparing $z_i$ with the other logits:

$$q_i = \frac{\exp(z_i/T)}{\sum_j \exp(z_j/T)} \qquad (1)$$

where $T$ is a temperature that is normally set to 1. Using a higher value for $T$ produces a softer probability distribution over classes.

**Worked example.** Consider a handwritten 2 and only the three classes "2, 3, 7". Suppose the cumbersome model produces the logits $v = (8, 4, 0)$.
We compute Eq. 1 in four steps: divide by $T$, exponentiate, sum, and divide each value by the sum.

In [ ]:
def softmax_T(z, T):
    """Eq. 1: softmax at temperature T, returning every intermediate step."""
    z = np.asarray(z, dtype='float64')
    a = z / T
    e = np.exp(a)
    s = e.sum()
    return a, e, s, e / s


v = [8.0, 4.0, 0.0]
for T in [1, 4]:
    a, e, s, p = softmax_T(v, T)
    print(f'T = {T}')
    print('  step 1  z/T        ', np.round(a, 4))
    print('  step 2  exp(z/T)   ', np.round(e, 4))
    print('  step 3  sum        ', round(s, 4))
    print('  step 4  probability', np.round(p, 4))
    print(f'  ratio of P(3) to P(7): {p[1] / p[2]:.2f}')

At $T = 1$ the probabilities of 3 and 7 are 0.0180 and 0.0003. Both are so close to zero that they have very little influence on the cross-entropy cost function during the transfer stage.
At $T = 4$ they become 0.2447 and 0.0900, and the information that this 2 resembles a 3 more than a 7 now affects learning.
We call the class probabilities produced by the cumbersome model at a high temperature **soft targets**, and the correct labels **hard targets**.

In [ ]:
labels = ['2', '3', '7']
fig, axes = plt.subplots(1, 2, figsize=(8, 2.8), sharey=True)
for ax, T in zip(axes, [1, 4]):
    p = softmax_T(v, T)[3]
    ax.bar(labels, p, color=['#1F4E79', '#C03030', '#59636E'])
    for k, val in enumerate(p):
        ax.text(k, val + 0.02, f'{val:.4f}', ha='center')
    ax.set_title(f'cumbersome model, T = {T}')
    ax.set_ylim(0, 1.1)
plt.show()

## 3. The distillation objective and its gradient

We use a weighted average of two different objective functions:

- The first is the cross entropy with the soft targets, $C_{soft} = -\sum_i p_i \log q_i$, computed using the same high temperature in the softmax of the distilled model as was used for generating the soft targets from the cumbersome model.
- The second is the cross entropy with the correct labels, $C_{hard}$, computed using exactly the same logits in the softmax of the distilled model but at a temperature of 1.

$$L = \alpha \, T^2 \, C_{soft} + (1 - \alpha) \, C_{hard}$$

Since the magnitudes of the gradients produced by the soft targets scale as $1/T^2$, we multiply them by $T^2$; this ensures that the relative contributions of the hard and soft targets remain roughly unchanged when the temperature is changed.
The best results are generally obtained with a considerably lower weight on the second objective function; here we use $\alpha = 0.9$.

Each case in the transfer set contributes a cross-entropy gradient with respect to each logit $z_i$ of the distilled model:

$$\frac{\partial C_{soft}}{\partial z_i} = \frac{1}{T}(q_i - p_i) \qquad (2)$$

**Worked example.** The same 2; the distilled model has the logits $z = (4, 2, 2)$, with $T = 4$ and $\alpha = 0.9$.

In [ ]:
T, alpha = 4.0, 0.9
z = np.array([4.0, 2.0, 2.0])
p = softmax_T(v, T)[3]          # soft targets of the cumbersome model
q = softmax_T(z, T)[3]          # distilled model at T = 4
q1 = softmax_T(z, 1)[3]         # distilled model at T = 1
y = np.array([1.0, 0.0, 0.0])   # the correct label is 2

c_soft = -(p * np.log(q)).sum()
c_hard = -(y * np.log(q1)).sum()
loss = alpha * T ** 2 * c_soft + (1 - alpha) * c_hard
print('soft targets p          ', np.round(p, 4))
print('distilled model q (T=4) ', np.round(q, 4))
print('distilled model q (T=1) ', np.round(q1, 4))
print(f'C_soft = {c_soft:.4f}   C_hard = {c_hard:.4f}')
print(f'L = {alpha} × {T:g}² × {c_soft:.4f} + {1 - alpha:.1f} × {c_hard:.4f} = {loss:.4f}')

g_soft = (q - p) / T            # Eq. 2
g_hard = q1 - y                 # gradient of the cross entropy with the correct labels
g = alpha * T ** 2 * g_soft + (1 - alpha) * g_hard
print()
print('soft-target gradient (q - p)/T, Eq. 2', np.round(g_soft, 4))
print('hard-target gradient q1 - y          ', np.round(g_hard, 4))
print('total gradient                       ', np.round(g, 4))

We now compute the same quantities with TensorFlow's automatic differentiation; the numbers must agree with those above:

In [ ]:
import tensorflow as tf

zt = tf.Variable(z)
with tf.GradientTape() as tape:
    qt = tf.nn.log_softmax(zt / T)
    q1t = tf.nn.log_softmax(zt)
    L = alpha * T ** 2 * -tf.reduce_sum(p * qt) + (1 - alpha) * -tf.reduce_sum(y * q1t)
print('loss from automatic differentiation    ', round(float(L), 4))
print('gradient from automatic differentiation', np.round(tape.gradient(L, zt).numpy(), 4))

**Why we multiply by $T^2$.** For the same logits, the gradient in Eq. 2 becomes smaller as the temperature rises. We compare $T = 1, 4, 20$:

In [ ]:
for t in [1, 4, 20]:
    gs = (softmax_T(z, t)[3] - softmax_T(v, t)[3]) / t
    print(f'T = {t:>2}  largest |gradient| {np.abs(gs).max():.6f}   multiplied by T² {np.abs(gs).max() * t * t:.4f}')

From $T = 4$ to $T = 20$ the gradient shrinks by a factor of about 20, close to $(20/4)^2 = 25$; multiplied by $T^2$, the two values are 0.85 and 1.05, which are of similar size.
The $1/T^2$ scaling holds in the high-temperature limit; $T = 1$ is outside it.

**Matching logits is a special case of distillation.** If the temperature is high compared with the magnitude of the logits, and the logits have been zero-meaned separately for each transfer case,
Eq. 2 simplifies to

$$\frac{\partial C_{soft}}{\partial z_i} \approx \frac{1}{N T^2}(z_i - v_i) \qquad (4)$$

where $N$ is the number of classes. So in the high-temperature limit, distillation is equivalent to minimizing $\frac{1}{2}(z_i - v_i)^2$.

In [ ]:
N, T = 3, 20.0
zc, vc = z - z.mean(), np.array(v) - np.mean(v)
exact = (softmax_T(zc, T)[3] - softmax_T(vc, T)[3]) / T
approx = (zc - vc) / (N * T ** 2)
print('zero-meaned logits  z =', np.round(zc, 4), '  v =', np.round(vc, 4))
print('Eq. 2, exact      ', np.round(exact, 5))
print('Eq. 4, approximate', np.round(approx, 5))

## 4. Training the cumbersome model

As in the paper, the cumbersome model is a net with two hidden layers of 1200 rectified linear hidden units.
It is strongly regularized using dropout, and the input images are jittered by up to two pixels in any direction.
The output layer produces logits; the temperature is applied separately.

In [ ]:
import keras
from keras import layers

keras.utils.set_random_seed(1)


def mlp(units, regularize=False, name='mlp'):
    """A net with two hidden layers of rectified linear units and 10 output logits.
    regularize=True adds jitter of up to two pixels and dropout."""
    inp = keras.Input((784,))
    x = inp
    if regularize:
        x = layers.Reshape((28, 28, 1))(x)
        x = layers.RandomTranslation(2 / 28, 2 / 28, fill_mode='constant')(x)
        x = layers.Flatten()(x)
        x = layers.Dropout(0.2)(x)
    for _ in range(2):
        x = layers.Dense(units, activation='relu')(x)
        if regularize:
            x = layers.Dropout(0.5)(x)
    out = layers.Dense(10)(x)
    return keras.Model(inp, out, name=name)


def test_errors(model, bias=None):
    """Return the number of test errors and the logits. bias is added to the logit of each class."""
    logits = model.predict(x_test, batch_size=2000, verbose=0)
    if bias is not None:
        logits = logits + bias
    return int((logits.argmax(1) != y_test).sum()), logits


cumbersome = mlp(1200, regularize=True, name='cumbersome')
cumbersome.compile(keras.optimizers.Adam(1e-3),
                   keras.losses.SparseCategoricalCrossentropy(from_logits=True))
cumbersome.fit(x_train, y_train, batch_size=128, epochs=30, verbose=0)
cumbersome_err, _ = test_errors(cumbersome)
print(f'cumbersome model: {cumbersome.count_params():,} parameters, {cumbersome_err} test errors')

# logits of the cumbersome model on the transfer set (the training set), used by every distilled model
v_train = cumbersome.predict(x_train, batch_size=2000, verbose=0)

## 5. Distillation versus training on hard targets

The small net has two hidden layers of 100 rectified linear hidden units and no regularization. We train it several times:

- **Hard targets only**: $\alpha = 0$, the normal way of training.
- **Distillation**: $\alpha = 0.9$ at temperatures of 1, 2, 4, 8, and 20.

The training loop implements the objective of Section 3 directly. Every run starts from the same initial weights and sees the training cases in the same order; only the objective differs.

In [ ]:
def train_small(units, T, alpha, mask=None, epochs=30, seed=2):
    """Train a small net with L = α T² C_soft + (1 − α) C_hard. mask removes cases from the transfer set."""
    keras.utils.set_random_seed(seed)
    small = mlp(units, name='small')
    opt = keras.optimizers.Adam(1e-3)
    x, yy, vv = (x_train, y_train, v_train) if mask is None else \
        (x_train[mask], y_train[mask], v_train[mask])
    ds = tf.data.Dataset.from_tensor_slices((x, yy, vv)).shuffle(len(x), seed=seed).batch(128)

    @tf.function
    def step(xb, yb, vb):
        with tf.GradientTape() as tape:
            zb = small(xb, training=True)
            pb = tf.nn.softmax(vb / T)                                     # soft targets
            c_soft = -tf.reduce_mean(tf.reduce_sum(pb * tf.nn.log_softmax(zb / T), 1))
            c_hard = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
                labels=yb, logits=zb))
            loss = alpha * T * T * c_soft + (1 - alpha) * c_hard
        grads = tape.gradient(loss, small.trainable_variables)
        opt.apply_gradients(zip(grads, small.trainable_variables))

    for _ in range(epochs):
        for xb, yb, vb in ds:
            step(xb, yb, vb)
    return small


TEMPS = [1, 2, 4, 8, 20]
results = {}
s = train_small(100, 1.0, 0.0)
results[(100, 'hard')] = test_errors(s)[0]
print(f'small net 100-100, hard targets only: {results[(100, "hard")]} test errors')
for t in TEMPS:
    s = train_small(100, float(t), 0.9)
    results[(100, t)] = test_errors(s)[0]
    print(f'small net 100-100, distilled at T = {t:>2}: {results[(100, t)]} test errors')

## 6. A much smaller net: intermediate temperatures work best

The paper reports that when the distilled net had 300 or more units in each of its two hidden layers, all temperatures above 8 gave fairly similar results,
but when this was radically reduced to 30 units per layer, temperatures in the range 2.5 to 4 worked significantly better than higher or lower temperatures.
At high temperatures distillation approaches the matching of logits, including logits that are much more negative than the average; a net that is much too small to capture all of the knowledge
does better when it can ignore them. We repeat the experiment with 30 units per layer:

In [ ]:
s = train_small(30, 1.0, 0.0)
results[(30, 'hard')] = test_errors(s)[0]
print(f'small net 30-30, hard targets only: {results[(30, "hard")]} test errors')
for t in TEMPS:
    s = train_small(30, float(t), 0.9)
    results[(30, t)] = test_errors(s)[0]
    print(f'small net 30-30, distilled at T = {t:>2}: {results[(30, t)]} test errors')

## 7. Omitting the digit 3 from the transfer set

We omit all examples of the digit 3 from the transfer set, so from the perspective of the distilled model, 3 is a mythical digit that it has never seen.
A net trained on hard targets alone can never output a 3, whereas the distilled model can still learn about 3s from the soft targets that the cumbersome model produces for other digits.

Most of the errors on 3s are caused by the fact that the learned bias for the 3 class is much too low. As in the paper, we increase this bias by the amount that optimizes overall performance on the test set;
because the test set is used to choose it, the resulting figures are optimistic.

In [ ]:
mask = y_train != 3
print('cases in the transfer set:', int(mask.sum()), '; of which 3s:', int((y_train[mask] == 3).sum()))


def threes_wrong(logits):
    return int(((logits.argmax(1) != y_test) & (y_test == 3)).sum())


for name, t, a in [('hard targets only', 1.0, 0.0), ('distilled at T = 8', 8.0, 0.9)]:
    s = train_small(100, t, a, mask=mask)
    err, logits = test_errors(s)
    best = None
    for b in np.arange(0, 15.01, 0.25):
        shift = np.zeros(10, 'float32')
        shift[3] = b
        e = int(((logits + shift).argmax(1) != y_test).sum())
        if best is None or e < best[0]:
            best = (e, b, threes_wrong(logits + shift))
    results[('no3', name)] = (err, threes_wrong(logits), best)
    print(f'{name}: {err} test errors, of which {threes_wrong(logits)} are on the 1010 threes')
    print(f'    bias for 3 increased by {best[1]:.2f}: {best[0]} test errors, of which {best[2]} are on 3s'
          f' ({100 * (1 - best[2] / 1010):.1f}% of the test 3s correct)')

## 8. Summary

The training set and the nets are smaller than in the paper, so the numbers of test errors differ from those reported there; what matters is whether the three effects are reproduced.

In [ ]:
print(f'{"model":<40}{"test errors":>12}')
print(f'{"cumbersome model 1200-1200":<40}{cumbersome_err:>12}')
for units in [100, 30]:
    print(f'{"small net %d-%d, hard targets only" % (units, units):<40}{results[(units, "hard")]:>12}')
    best_t = min(TEMPS, key=lambda t: results[(units, t)])
    print(f'{"small net %d-%d, distilled, best T = %d" % (units, units, best_t):<40}{results[(units, best_t)]:>12}')

fig, ax = plt.subplots(figsize=(7, 3.2))
for units, color in [(100, '#1F4E79'), (30, '#C03030')]:
    ax.plot(TEMPS, [results[(units, t)] for t in TEMPS], 'o-', color=color,
            label=f'small net {units}-{units}, distilled')
    ax.axhline(results[(units, 'hard')], color=color, ls='--',
               label=f'small net {units}-{units}, hard targets only')
ax.set_xscale('log')
ax.set_xticks(TEMPS)
ax.set_xticklabels([str(t) for t in TEMPS])
ax.set_xlabel('temperature T')
ax.set_ylabel('test errors (of 10,000)')
ax.legend(fontsize=8)
plt.show()

| Effect reported in the paper | Section of this notebook |
|---|---|
| Distillation gives fewer test errors than training on hard targets | Section 5 |
| When the distilled net is very small, intermediate temperatures work best | Section 6 |
| With the right bias, the distilled model gets most test 3s correct despite never having seen a 3 | Section 7 |

**Exercises**

1. In Section 3, set the logits of the distilled model to $(8, 4, 0)$, identical to those of the cumbersome model. What is the soft-target gradient, and why?
2. In Section 5, change $\alpha$ to 0.5. How does this affect the result of distillation?
3. In Section 7, omit the digit 7 instead and repeat the experiment.

---

Reference: G. Hinton, O. Vinyals, J. Dean, "Distilling the Knowledge in a Neural Network," arXiv:1503.02531, 2015.
Data: MNIST handwritten digits (Y. LeCun, C. Cortes, C. J. C. Burges); this module uses 10,000 training cases and the full test set.